In [2]:
import xarray as xr
import glob
import os
import pandas as pd

#### Merge these as a compiled nc file

In [3]:
mergedE = xr.open_mfdataset('../../../../data/finaldatasets/covariates/Covariates/GLEAM/actualevaporation/*.nc')
mergedEs = xr.open_mfdataset('../../../../data/finaldatasets/covariates/Covariates/GLEAM/evaporativestress/*.nc')
mergedSM = xr.open_mfdataset('../../../../data/finaldatasets/covariates/Covariates/GLEAM/rootsoilmoisture/*.nc')
mergedT = xr.open_mfdataset('../../../../data/finaldatasets/covariates/Covariates/GLEAM/transpiration/*.nc')

Data overview

In [4]:
# Summary function for all datasets
datasets = {
   'Actual Evaporation': mergedE,
   'Evaporative Stress': mergedEs, 
   'Root Soil Moisture': mergedSM,
   'Transpiration': mergedT
}

for name, ds in datasets.items():
   print(f"\n{'='*50}")
   print(f"DATASET: {name}")
   print(f"{'='*50}")
   print(ds)
   print(f"\nTime range: {ds.time.min().values} to {ds.time.max().values}")
   print(f"Total time steps: {len(ds.time)}")
   print(f"Spatial dimensions: {ds.dims}")


DATASET: Actual Evaporation
<xarray.Dataset>
Dimensions:  (time: 300, lat: 1800, lon: 3600)
Coordinates:
  * time     (time) datetime64[ns] 2000-01-31 2000-02-29 ... 2024-12-31
  * lat      (lat) float64 89.95 89.85 89.75 89.65 ... -89.75 -89.85 -89.95
  * lon      (lon) float64 -179.9 -179.8 -179.8 -179.7 ... 179.7 179.8 179.9
Data variables:
    E        (time, lat, lon) float32 dask.array<chunksize=(12, 1800, 3600), meta=np.ndarray>
Attributes:
    Dataset:      Global Land Evaporation Amsterdam Model
    Version:      4.2a
    Authors:      Hydro-Climate Extremes Lab (H-CEL)
    Institution:  Ghent University
    Contact:      info@gleam.eu
    Reference1:   Miralles, D.G. et al. 2011: Global land-surface evaporation...
    Reference2:   Miralles, D. G., Bonte, O., Koppa, A., Villanueva, O. B., T...

Time range: 2000-01-31T00:00:00.000000000 to 2024-12-31T00:00:00.000000000
Total time steps: 300
Spatial dimensions: Frozen({'time': 300, 'lat': 1800, 'lon': 3600})

DATASET: Evaporat

Save as NetCDF files

In [5]:
mergedE.to_netcdf('../../../../data/finaldatasets/covariates/Covariates/evaporation.nc')
mergedEs.to_netcdf('../../../../data/finaldatasets/covariates/Covariates/evap_stress.nc') 
mergedSM.to_netcdf('../../../../data/finaldatasets/covariates/Covariates/soil_moisture.nc')
mergedT.to_netcdf('../../../../data/finaldatasets/covariates/Covariates/transpiration.nc')

lowering spacial resolution of and compiling NDVI nc files

In [4]:
#Examine metadata of NDVI file
test = xr.open_dataset('../../../../data/finaldatasets/covariates/Covariates/NVMI/2004/c_gls_NDVI_200401010000_GLOBE_VGT_V3.0.1.nc')

# Examine the dataset
print(test)


<xarray.Dataset>
Dimensions:    (time: 1, lat: 15680, lon: 40320)
Coordinates:
  * lat        (lat) float64 80.0 79.99 79.98 79.97 ... -59.97 -59.98 -59.99
  * lon        (lon) float64 -180.0 -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
  * time       (time) datetime64[ns] 2004-01-01
Data variables:
    NDVI       (time, lat, lon) float32 ...
    NDVI_unc   (time, lat, lon) float32 ...
    NOBS       (time, lat, lon) float32 ...
    QFLAG      (time, lat, lon) float32 ...
    TIME_GRID  (time, lat, lon) float32 ...
    crs        |S1 ...
Attributes: (12/19)
    Conventions:          CF-1.6
    processing_level:     L3
    identifier:           urn:cgls:global:ndvi_v3_1km:NDVI_200401010000_GLOBE...
    institution:          VITO NV
    time_coverage_end:    2004-01-10T23:59:59Z
    source:               Derived from EO satellite imagery
    ...                   ...
    time_coverage_start:  2004-01-01T00:00:00Z
    platform:             SPOT-5
    title:                10-daily Normalize

In [3]:
# === CONFIGURATION ===
base_dir = "../../../../data/finaldatasets/covariates/Covariates/NVMI/"
output_dir = "../../../../data/finaldatasets/covariates/Covariates/"
final_output = "NDVI_10km_2004_2020.nc"
os.makedirs(output_dir, exist_ok=True)
years = range(2004, 2021)

# === STEP 1: Process Each Year Individually ===
for year in years:
    print(f"📆 Processing year: {year}")
    
    file_pattern = os.path.join(base_dir, str(year), "*.nc")
    files = sorted(glob.glob(file_pattern))
    
    if not files:
        print(f"⚠️ No files for {year}, skipping.")
        continue

    try:
        # Open and chunk with Dask for memory safety
        ds = xr.open_mfdataset(
            files,
            combine="by_coords",
            chunks={"lat": 1000, "lon": 1000}
        )[["NDVI"]]

        # Coarsen to 10km (10x10 grid cells)
        ds_coarse = ds.coarsen(lat=10, lon=10, boundary="trim").mean()

        # Compute and save one file per year
        out_path = os.path.join(output_dir, f"NDVI_10km_{year}.nc")
        ds_coarse.compute().to_netcdf(out_path)

        print(f"✅ Saved: {out_path}")

    except Exception as e:
        print(f"❌ Error in year {year}: {e}")

# === STEP 2: Merge All Coarsened Yearly Files ===
print("\n🔁 Merging yearly files into one...")
all_years = sorted(glob.glob(os.path.join(output_dir, "NDVI_10km_*.nc")))

merged = xr.open_mfdataset(
    all_years,
    combine="by_coords",
    chunks={"lat": 1000, "lon": 1000}
)

merged.to_netcdf(final_output)
print(f"🎉 All done! Final file saved as: {final_output}")

📆 Processing year: 2004
✅ Saved: ../../../../data/finaldatasets/covariates/Covariates/NDVI_10km_2004.nc
📆 Processing year: 2005
✅ Saved: ../../../../data/finaldatasets/covariates/Covariates/NDVI_10km_2005.nc
📆 Processing year: 2006
✅ Saved: ../../../../data/finaldatasets/covariates/Covariates/NDVI_10km_2006.nc
📆 Processing year: 2007
✅ Saved: ../../../../data/finaldatasets/covariates/Covariates/NDVI_10km_2007.nc
📆 Processing year: 2008
✅ Saved: ../../../../data/finaldatasets/covariates/Covariates/NDVI_10km_2008.nc
📆 Processing year: 2009
✅ Saved: ../../../../data/finaldatasets/covariates/Covariates/NDVI_10km_2009.nc
📆 Processing year: 2010
✅ Saved: ../../../../data/finaldatasets/covariates/Covariates/NDVI_10km_2010.nc
📆 Processing year: 2011
✅ Saved: ../../../../data/finaldatasets/covariates/Covariates/NDVI_10km_2011.nc
📆 Processing year: 2012
✅ Saved: ../../../../data/finaldatasets/covariates/Covariates/NDVI_10km_2012.nc
📆 Processing year: 2013
✅ Saved: ../../../../data/finaldatasets/

Check if Files look ok 

In [4]:
test = xr.open_dataset('../../../../data/finaldatasets/covariates/Covariates/NDVI_10km_2004_2020.nc')


print(test)

<xarray.Dataset>
Dimensions:  (time: 594, lat: 1568, lon: 4032)
Coordinates:
  * lat      (lat) float64 79.96 79.87 79.78 79.69 ... -59.77 -59.86 -59.95
  * lon      (lon) float64 -180.0 -179.9 -179.8 -179.7 ... 179.8 179.9 180.0
  * time     (time) datetime64[ns] 2004-01-01 2004-01-11 ... 2020-06-21
Data variables:
    NDVI     (time, lat, lon) float32 ...
Attributes: (12/19)
    Conventions:          CF-1.6
    processing_level:     L3
    identifier:           urn:cgls:global:ndvi_v3_1km:NDVI_200401010000_GLOBE...
    institution:          VITO NV
    time_coverage_end:    2004-01-10T23:59:59Z
    source:               Derived from EO satellite imagery
    ...                   ...
    time_coverage_start:  2004-01-01T00:00:00Z
    platform:             SPOT-5
    title:                10-daily Normalized Difference Vegetation Index 1KM...
    archive_facility:     VITO
    parent_identifier:    urn:cgls:global:ndvi_v3_1km
    history:              2021-02-03: Processing line NDVI


Compile to show monthly time series to be used in R pipeline 

In [3]:
# === Load dataset with Dask chunking ===
ds = xr.open_dataset(
    "../../../../data/finaldatasets/covariates/Covariates/NDVI_10km_2004_2020.nc",
    chunks={"lat": 1000, "lon": 1000}  # adjust if needed
)

# === Assign proper time coordinate ===
ndvi_time = pd.date_range(start="2004-01-01", periods=ds.dims["time"], freq="10D")
ds = ds.assign_coords(time=("time", ndvi_time))

# === Coarsen every 3 x 10-day steps into monthly ===
ds_monthly = ds.coarsen(time=3, boundary="trim").mean()

# === Compute safely BEFORE saving ===
ds_monthly = ds_monthly.compute()

# === Save with compression and chunking ===
ds_monthly.to_netcdf(
    "../../../../data/finaldatasets/covariates/Covariates/NDVI_monthly_10km_2004_2020.nc",
    encoding={
        "NDVI": {
            "zlib": True,
            "complevel": 4,
            "chunksizes": (36, 500, 500)
        }
    }
)

print("✅ NDVI monthly average saved (chunked + compressed)")

✅ NDVI monthly average saved (chunked + compressed)
